# Chat Demo

Chat follow-up is grounded in the assembled evidence object. It requires a configured LLM provider and does not use a local template fallback.

In [1]:
from pathlib import Path
import os
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", message="IProgress not found.*")

repo_root = Path.cwd()
if not (repo_root / "src" / "pxfquery").exists() and (repo_root.parent / "src" / "pxfquery").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Optional: load local environment variables for the LLM provider.
for env_file in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.home() / ".env"]:
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

from pxfquery import PxFQuery

pxf = PxFQuery()
print("PxFquery", pxf.version)
resource_status = pxf.resources.status()
resource_info = resource_status.to_dict() if hasattr(resource_status, "to_dict") else dict(resource_status)
print("Resource status:")
print({
    "available": resource_info.get("available"),
    "source": resource_info.get("source"),
    "version": resource_info.get("version"),
    "available_file_count": len(resource_info.get("available_files") or {}),
})


PxFquery 0.5.12.dev0
Resource status:
{'available': True, 'source': 'manifest', 'version': 'v20260628', 'available_file_count': 18}


In [2]:
question = (
    "In melanoma models, which genetic knockdown perturbations are linked to suppression "
    "of EMT programs?"
)

print("Question:")
print(question)

qdata = pxf.tl.parse(question, top_n=10)
pxf.tl.answer(qdata)
print("\nBase answer:")
print(pxf.get.answer(qdata))

Question:
In melanoma models, which genetic knockdown perturbations are linked to suppression of EMT programs?


[Parsing] start


[Parsing] done | mode=reverse; context=melanoma models; time=1.89s
[Matching] start


[Matching] done | matches=3; time=3.34s
[Matrix] start


[Matrix] done | profiles=3; skipped=0; time=12.57s
[Evidence] start



Base answer:
Answer
The top candidate for suppressing EMT programs in melanoma models is HAVCR1 knockdown, as it shows the strongest and most consistent functional match across multiple shRNA profiles. Other strong candidates include LRSAM1, DGKD, OTUD7A, and NME9, all of which are recommended for inhibition or knockout and show high match scores. Weaker candidates include IL2RB, SRSF3, ATP7A, POLR2F, ZBTB48, BRAP, and TNFRSF11A, which have lower or less consistent support.

Analysis source: pxfquery 0.5.12.dev0
Evidence: inspect `answer.tables["route_summary"]`, `answer.tables["ranked_results"]`, and `answer.tables["route_target_functions"]`.
Figures: call `pxf.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.


[Evidence] done | status=ready; time=7.16s


In [3]:
follow_up = (
    "Using only the current evidence object, identify the best-supported knockdown candidates "
    "and note any uncertainty in target identity or model matching."
)

print("Follow-up:")
print(follow_up)

pxf.tl.chat(qdata, follow_up, print_response=False)
print("\nChat response:")
print(pxf.get.chat(qdata))

Follow-up:
Using only the current evidence object, identify the best-supported knockdown candidates and note any uncertainty in target identity or model matching.



Chat response:
The best-supported knockdown candidates for suppressing EMT programs in melanoma models are HAVCR1, LRSAM1, DGKD, OTUD7A, and NME9, all recommended for inhibition or knockout with high match scores across multiple shRNA profiles. Uncertainty exists because exact cell support is false (all candidates are based on a concept representative cell A375, not an exact melanoma model match), and literature evidence is disabled, so no PubMed validation is available.


In [4]:
print("Chat history:")
display(pxf.get.chat_history(qdata))

Chat history:


[{'user': 'Using only the current evidence object, identify the best-supported knockdown candidates and note any uncertainty in target identity or model matching.',
  'assistant': 'The best-supported knockdown candidates for suppressing EMT programs in melanoma models are HAVCR1, LRSAM1, DGKD, OTUD7A, and NME9, all recommended for inhibition or knockout with high match scores across multiple shRNA profiles. Uncertainty exists because exact cell support is false (all candidates are based on a concept representative cell A375, not an exact melanoma model match), and literature evidence is disabled, so no PubMed validation is available.',
  'cited_tables': ['ranked_results', 'route_summary', 'claim_rules'],
  'warnings': ['Literature evidence is disabled; no PubMed support.',
   'Exact cell support is false for all candidates; evidence is based on a concept representative cell (A375).'],
  'provider_evidence': {'provider': 'deepseek',
   'base_url': 'https://api.deepseek.com/v1',
   'mode